In [1]:
from typing import Optional
from ollama import chat # type: ignore
from pydantic import BaseModel, Field # type: ignore
import json
import tiktoken
import pandas as pd
import glob

In [2]:
def load_evidence(fn):
    ds_name = fn.split("/")[-3]
    df = pd.read_json(fn)
    # Unwrap the 'source' column (contains a dictionary) into separate columns
    source_df = df['source'].apply(pd.Series)
    extracted_df = df['extracted'].apply(pd.Series).add_prefix('extracted_')
    # derived_df = df['derived'].apply(pd.Series).add_prefix('derived_')
    # Combine the original DataFrame with the unwrapped columns
    # df = pd.concat([df.drop(columns=['source', 'extracted', 'derived']), source_df, extracted_df, derived_df], axis=1)
    df = pd.concat([df.drop(columns=['source', 'extracted']), source_df, extracted_df], axis=1)
    df["ds"] = ds_name
    del df["derived"]
    return df

In [3]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Permissive class object (allows for missing fields)
class MarkerGene(BaseModel):
    marker_gene_name: Optional[str] = Field(
        None, 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: Optional[str] = Field(
        None, 
        description="The name of the cell type for which this gene serves as a marker."
    )

class MarkerGeneList(BaseModel):
    genes: List[MarkerGene]  # A list of marker genes extracted from the input text

# Restrictive class object does not allow for missing fields
class MarkerGeneStrict(BaseModel):
    marker_gene_name: str = Field(
        ..., 
        description="The name of the marker gene that is associated with a specific cell type."
    )
    cell_type_name: str = Field(
        ..., 
        description="The name of the cell type for which this gene serves as a marker."
    )

class MarkerGeneListStrict(BaseModel):
    genes: List[MarkerGeneStrict]  # A list of marker genes extracted from the input text

In [4]:
def extract_genes(user_content, system_prompt, data_model, model="llama3.2"):
    response = chat(
        messages=[
            {
                'role': 'system',
                'content': system_prompt,
            },
            {
                'role': 'user',
                'content': user_content,
            }
        ],
        model = model,
        format = data_model.model_json_schema(),
        options={'temperature': 0},  # Make responses more deterministic

    )
    genes = data_model.model_validate_json(response.message.content)
    return genes.model_dump()["genes"]

In [5]:
system_prompt = """
You are an expert in genomics analyzing scientific literature to extract marker genes for different cell types. 

Your goal is to identify and structure marker gene data from the given text. For each marker gene mentioned, extract:
- The **gene name** (marker_gene_name).
- The **cell type** it is associated with (cell_type_name).

The data must be extracted as written in the text, without any modifications.

Return the results in **structured JSON format** with the following schema:
{
    "genes": [
        {
            "cell_type_name": "Neuron",
            "marker_gene_name": "GeneX",
        },
        ...
    ]
}
"""
user_content = """
After doublet removal and quality filtering, we considered a total of 197,721 cells (106,469 from PG and 91,252 from ING), 
identifying all cell types observed in human WAT (Fig. 2c, d, Supplementary Table 2) with the addition of 
distinct male and female epithelial populations (Dcdc2a+ and Erbb4+, respectively)
"""

# system_prompt = "You are a genomics researcher trying to discover different cell types and what genes they express, aka find the marker genes within the given sentence."

In [6]:
fns = glob.glob("../../data/*/evidence_human/evidence.json")

In [7]:
dfs = []
for fn in fns:
    dfs.append(load_evidence(fn))
df = pd.concat(dfs)

In [8]:
df.groupby("source_type").size()

source_type
image     378
img      2385
text     1066
dtype: int64

In [15]:
df.query('source_type != "text"').groupby("ds")["source_type"].value_counts().sort_values()

ds                        source_type
pancreas_Segerstolpe2016  img             12
adipose_Jaitin2019        img             14
adipose_Vijay2019         img             26
bladder_Yu2019            img             38
cortex_ Booeshaghi2021    img             40
testis_Shamis2020         img             66
liver_Guillams2022        image           69
ovary_Wagner2020          image           80
adipose_Hildreth2021      img             98
placenta_Liu2018          img             98
lung_Adams2020            img            111
yolksac_Goh2023           img            137
kidney_Wu2018             img            228
adipose_Massier2023       image          229
bone_He2021               img            327
eye_Gautam2021            img            348
adipose_Emont2022         img            356
heart_Tucker2020          img            486
Name: count, dtype: int64

In [ ]:
tdeg = df.query("source_type == 'text'").copy()

In [ ]:
uniq_src = tdeg.source_rationale.unique()

In [ ]:
genes = []
for idx, u in enumerate(uniq_src):
    if idx % 25 == 0:
        print(f"Processing {idx}/{len(uniq_src)}")
    genes.append({u: extract_genes(u, system_prompt, MarkerGeneListStrict, model="llama3.2")})

Processing 0/289
Processing 25/289
Processing 50/289
Processing 75/289
Processing 100/289
Processing 125/289
Processing 150/289
Processing 175/289
Processing 200/289
Processing 225/289
Processing 250/289
Processing 275/289


In [ ]:
pd.DataFrame([{"num_entries": len(g[list(g.keys())[0]]), "source": list(g.keys())[0]} for g in genes]).sort_values("num_entries", ascending=False).iloc[0]["source"]

'However, we identify many genes that are either expressed in all three species (i.e., FMR1NB, ZCWPW1, DPEP3, IQBP1, and CALR), in primates only (BEND2, ZRANB2, PAGE4, ZNFX1, HLTF, and ZBED5), or are singularly expressed in mouse (Swt1, Tgfbr1, Rhox13, PhTF1, and Fmr1), macaque (ZNF850, MAGEB17, ZMYM1, and TCEA1), and human (PRDM7, full VCX family, ERBB3, ERVK13-1, SSX3, and TRIML2)'

In [ ]:
records = []
for item in genes:
    for sentence, objects in item.items():
        for obj in objects:
            record = {'sentence': sentence, **obj}
            records.append(record)

In [ ]:
pd.DataFrame(records)

,sentence,marker_gene_name,cell_type_name,tissue_source,species
0,Monocyte 1 strongly expressed FCGR3A (CD16) an...,FCGR3A,monocyte,blood,human
1,"Interestingly, ABCA1, which mediates sterol ef...",ABCA1,dendritic cells (DCs),monocyte 1,human
2,Although both cells clusters expressed typical...,SDC3,monocyte,biopsy dataset,human
3,Although both cells clusters expressed typical...,ABCA1,monocyte,biopsy dataset,human
4,Although both cells clusters expressed typical...,APOE,DC maturation marker,biopsy dataset,human
...,...,...,...,...,...
1051,"Furthermore, the stromal cells also expressed ...",ANGPTL1,stromal cells,stromal cells,unknown
1052,"Furthermore, the stromal cells also expressed ...",ANGPTL2,stromal cells,stromal cells,unknown
1053,"Furthermore, the stromal cells also expressed ...",ANGPTL4,stromal cells,stromal cells,unknown
1054,"Furthermore, the stromal cells also expressed ...",CTGF,stromal cells,stromal cells,unknown


In [ ]:
tdeg.groupby("source_rationale")["source_type"].count()#.sort_values(ascending=False)

source_rationale
 CD74 was highly enriched in sfC13 and ofC10                                                                                                                                                                                                                                                                                                      1
 Finally, we noted that CALM2 was negatively correlated with BMI in endocrine (α, β, γ, and δ) and ductal cells                                                                                                                                                                                                                                   5
(1) NGFR+ cranial neural crest (NC) cells (cluster 1) that highly expressed NES                                                                                                                                                                                                                                